# Clinical Data Lakehouse — Synthea → Medallion → SDTM

A medallion-architecture lakehouse built on **Databricks** (Free Edition, serverless)
over **Synthea** synthetic patient data, then mapped into **CDISC SDTM** and validated
with the open-source **CDISC CORE** conformance engine.

**Pipeline:** raw CSV → **Bronze** (Delta, untouched) → **Silver** (typed, conformed,
HIPAA Safe Harbor de-identified) → **Gold** (ML feature table + cohort analytics table)
→ **SDTM** (RWD → CDISC SDTMIG v3.4: DM, VS, LB, CM, PR, MH).

See [`README.md`](../README.md) for architecture and design rationale, and
[`VALIDATION.md`](../VALIDATION.md) for the CDISC CORE conformance triage.

> Synthetic data only — no real PHI. Generated with a fixed seed for reproducibility
> (see README). This notebook was authored and run in Databricks; outputs below are
> from a 100-patient sample.


## Bronze layer — raw ingestion

Land the six core Synthea domains as Delta tables **as-is**. Everything is read as a
string (`inferSchema=false`) so the source is preserved exactly — typing is Silver's job.
The only additions are lineage metadata (`_ingested_at`, `_source_file`).


In [1]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw;

In [2]:
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "workspace"   # change if your catalog panel showed something else
SCHEMA  = "bronze"
RAW     = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

tables = ["patients", "encounters", "conditions",
          "medications", "observations", "procedures"]

for t in tables:
    df = (spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "false")   # keep everything as raw strings — typing is silver's job
            .load(f"{RAW}/{t}.csv"))

    df = (df.withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", lit(f"{t}.csv")))

    (df.write.format("delta").mode("overwrite")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{t}"))

    print(f"wrote {CATALOG}.{SCHEMA}.{t}: {df.count()} rows")

wrote workspace.bronze.patients: 112 rows
wrote workspace.bronze.encounters: 5906 rows
wrote workspace.bronze.conditions: 3975 rows
wrote workspace.bronze.medications: 4096 rows
wrote workspace.bronze.observations: 73883 rows
wrote workspace.bronze.procedures: 17002 rows


In [3]:
%sql
SELECT * FROM workspace.bronze.patients LIMIT 10;

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,_ingested_at,_source_file
a35dfdc9-1d97-63e0-a0b6-3e97b6fd5fe1,1995-07-04,null,999-47-1442,S99967996,X82997617X,Mr.,Joel444,Isaias604,Hand679,null,null,M,white,nonhispanic,M,North Brookfield Massachusetts US,370 Moore Flat,Boston,Massachusetts,Suffolk County,25025,02108,42.38455212619432,-71.06049007739105,50659.61,0.00,534421,2026-06-24T14:30:02.697Z,patients.csv
5393ea7f-1fea-4064-9df4-460a2f662d07,1974-02-25,1986-03-04,999-24-7706,null,null,null,Miguel815,null,Bashirian201,null,null,null,white,nonhispanic,M,Methuen Massachusetts US,339 Stoltenberg Village,Dalton,Massachusetts,Berkshire County,null,00000,42.4700163962609,-73.16577471299114,3091.72,27765.55,3229,2026-06-24T14:30:02.697Z,patients.csv
595f9fb9-fcf0-e5bc-239a-e86a96db6211,1971-10-23,null,999-58-2324,S99980593,X54544021X,Mr.,Lucius907,null,Emard19,null,null,M,white,nonhispanic,M,Stoughton Massachusetts US,609 King Promenade Apt 41,Lowell,Massachusetts,Middlesex County,25017,01854,42.6251419384755,-71.33235620401317,8494.16,116265.06,3928,2026-06-24T14:30:02.697Z,patients.csv
eed8a921-eac8-0778-4113-255f4e35506a,1962-10-25,null,999-86-5479,S99966124,X48077728X,Mrs.,Kimi714,Katerine813,Cummings51,null,Nienow652,W,black,nonhispanic,F,Marshfield Massachusetts US,413 Schiller Green Suite 85,Boston,Massachusetts,Suffolk County,25025,02134,42.394648625457,-71.08231961066721,563455.39,315100.37,45680,2026-06-24T14:30:02.697Z,patients.csv
5c9f20d5-8361-b331-9267-5303ca5b136a,1969-09-19,null,999-50-6764,S99945350,X10720646X,Mr.,Andrés117,Hernán834,Olivo261,null,null,M,white,hispanic,M,Caracas Capital District VE,1036 Hettinger Ville,Chicopee,Massachusetts,Hampden County,25013,01020,42.16251064874955,-72.54557988719839,174470.93,11038.65,88407,2026-06-24T14:30:02.697Z,patients.csv
d545798e-09a4-ef89-abc5-9f62dc5a5095,1997-10-19,null,999-28-1649,S99979150,X41702617X,Mrs.,Ashely524,Maybelle917,Schowalter414,null,Brekke496,M,white,nonhispanic,F,Nantucket Massachusetts US,873 Mayer Avenue,Shrewsbury,Massachusetts,Worcester County,null,00000,42.29342226674201,-71.68984373787688,158945.24,295183.95,503848,2026-06-24T14:30:02.697Z,patients.csv
72b5373b-e9f1-db2d-6c43-4506fb4b3e3f,1961-12-29,null,999-14-7567,S99923739,X47439218X,Mrs.,Marta91,null,Godínez202,null,Aguilera202,M,white,hispanic,F,Tegucigalpa Francisco Morazán HN,991 Aufderhar Trailer,Lynn,Massachusetts,Essex County,25009,01901,42.42650323579352,-70.91792050617427,13887.73,947222.71,5779,2026-06-24T14:30:02.697Z,patients.csv
3c4b499b-090b-c6fa-28d7-b56c6056d0a2,1946-04-24,null,999-80-6276,S99941854,X11051445X,Mrs.,Louie190,Danita413,West559,null,Botsford977,M,white,nonhispanic,F,Brookline Massachusetts US,135 Tremblay Fort,South Yarmouth,Massachusetts,Barnstable County,25001,02664,41.70084416937746,-70.19108338111833,647989.08,417494.44,67198,2026-06-24T14:30:02.697Z,patients.csv
b9b2d24d-b969-9415-d821-b90a382311af,1974-02-25,null,999-16-6295,S99934260,X61241399X,Mr.,Benton624,Errol226,Ebert178,null,null,M,white,nonhispanic,M,Andover Massachusetts US,1001 Bosco Extension Suite 32,Dalton,Massachusetts,Berkshire County,null,00000,42.477567171322335,-73.13663290187931,9978.55,1136320.91,3229,2026-06-24T14:30:02.697Z,patients.csv
10ca2e4f-9402-1da8-54f1-059e62503949,2004-06-19,null,999-29-1028,S99935251,X27667915X,Ms.,Lakeisha206,Crista774,Hessel84,null,null,null,white,nonhispanic,F,New Bedford Massachusetts US,937 Kreiger Road Suite 10,Malden,Massachusetts,Middlesex County,25017,02155,42.464107720187265,-71.07710569594408,62085.69,5595.10,35460,2026-06-24T14:30:02.697Z,patients.csv


## Silver layer — typed, conformed, de-identified

Cast strings to real types, conform names, and de-identify. The `patients` table applies
**HIPAA Safe Harbor** (45 CFR §164.514(b)(2)): direct identifiers dropped, geography
reduced to state + 3-digit ZIP (17 restricted prefixes → `000`), dates reduced to year,
ages > 89 aggregated to a `90+` cohort with birth year suppressed. Clinical/geographic
codes (ZIP, FIPS, SNOMED, LOINC, RxNorm) deliberately stay strings to preserve leading
zeros and code fidelity. Durations (e.g. length of stay) are derived before dates are
dropped — *a duration is not a date* under Safe Harbor.


In [4]:
from pyspark.sql import functions as F

CATALOG = "workspace"   # match your bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

# HHS Safe Harbor: 3-digit ZIP prefixes for areas <=20,000 people -> '000'
RESTRICTED_ZIP3 = ["036","059","063","102","203","556","692","790","821",
                   "823","830","831","878","879","884","890","893"]

df = spark.table(f"{CATALOG}.bronze.patients")
df = df.toDF(*[c.lower() for c in df.columns])

# dates parsed ONLY to derive age; full dates are not carried forward
df = (df.withColumn("birthdate", F.to_date("birthdate"))
        .withColumn("deathdate", F.to_date("deathdate")))

# age at death if deceased, else as of today
ref = F.coalesce(F.col("deathdate"), F.current_date())
age_raw = F.floor(F.months_between(ref, F.col("birthdate")) / 12).cast("int")

# zip3 with restricted-prefix suppression
z3 = F.substring(F.col("zip"), 1, 3)
z3 = F.when(z3.isin(RESTRICTED_ZIP3), F.lit("000")).otherwise(z3)

silver = (df
    .withColumn("age_raw", age_raw)
    .withColumn("age_90_plus", F.col("age_raw") >= 90)
    # date rule: year only, and suppress birth_year for the 90+ cohort
    .withColumn("age", F.when(F.col("age_raw") >= 90, F.lit(90)).otherwise(F.col("age_raw")))
    .withColumn("birth_year", F.when(F.col("age_raw") >= 90, F.lit(None).cast("int"))
                                .otherwise(F.year("birthdate")))
    .withColumn("death_year", F.year("deathdate"))
    .withColumn("is_deceased", F.col("deathdate").isNotNull())
    .withColumn("zip3", z3)
    .withColumnRenamed("id", "patient_id")
    # project to the de-identified column set (drops names, ssn, license,
    # passport, address, city, county, fips, lat/lon, full dates, birthplace)
    .select(
        "patient_id", "birth_year", "age", "age_90_plus",
        "is_deceased", "death_year",
        "marital", "race", "ethnicity", "gender",
        "state", "zip3",
        F.col("healthcare_expenses").cast("decimal(12,2)").alias("healthcare_expenses"),
        F.col("healthcare_coverage").cast("decimal(12,2)").alias("healthcare_coverage"),
        F.col("income").cast("int").alias("income"),
        "_ingested_at",
    ))

(silver.write.format("delta").mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.silver.patients"))

print(f"silver.patients (Safe Harbor de-identified): {silver.count()} rows, {len(silver.columns)} cols")
silver.printSchema()

silver.patients (Safe Harbor de-identified): 112 rows, 16 cols
root
 |-- patient_id: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- age_90_plus: boolean (nullable = true)
 |-- is_deceased: boolean (nullable = false)
 |-- death_year: integer (nullable = true)
 |-- marital: string (nullable = true)
 |-- race: string (nullable = true)
 |-- ethnicity: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip3: string (nullable = true)
 |-- healthcare_expenses: decimal(12,2) (nullable = true)
 |-- healthcare_coverage: decimal(12,2) (nullable = true)
 |-- income: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



In [5]:
%sql
SELECT patient_id, birth_year, age, age_90_plus, is_deceased, death_year,
       state, zip3, race, ethnicity, income
FROM workspace.silver.patients
ORDER BY age DESC
LIMIT 15;

patient_id,birth_year,age,age_90_plus,is_deceased,death_year,state,zip3,race,ethnicity,income
da4d2cd8-56df-e44e-4ee2-5fb115d42d0a,1941,85,false,false,null,Massachusetts,021,white,hispanic,40040
3c4b499b-090b-c6fa-28d7-b56c6056d0a2,1946,80,false,false,null,Massachusetts,026,white,nonhispanic,67198
40e66c47-f6e8-c2e7-6be6-dbc4b4fabe8d,1946,80,false,false,null,Massachusetts,018,white,hispanic,90106
552461fe-485c-659e-a02a-3f43b6f4d209,1946,79,false,false,null,Massachusetts,025,white,nonhispanic,65295
078df27f-f693-a4cf-9f23-872b9e3ab3d2,1941,79,false,true,2020,Massachusetts,021,white,hispanic,40040
4baf953f-897a-3bf2-4bfd-1cb98984249c,1947,78,false,false,null,Massachusetts,011,white,nonhispanic,5376
b23188ac-9529-2450-e0b7-58adb2b38de6,1949,77,false,false,null,Massachusetts,010,white,nonhispanic,45097
7bbaa47e-d029-bf4f-3237-ddd955c0a584,1949,76,false,false,null,Massachusetts,019,black,hispanic,15648
ea8df98b-6ed8-2f53-9f6a-80af4fa3bd0f,1951,74,false,false,null,Massachusetts,026,white,nonhispanic,47252
e210fadc-336e-f8b7-8934-f7b36b8e288b,1954,71,false,false,null,Massachusetts,021,asian,nonhispanic,35382


In [6]:
from pyspark.sql import functions as F

CATALOG = "workspace"

def read_bronze(name):
    df = spark.table(f"{CATALOG}.bronze.{name}")
    return df.toDF(*[c.lower() for c in df.columns])   # normalize to snake-ish

def apply_casts(df, casts):                            # {column: type}
    for col, typ in casts.items():
        df = df.withColumn(col, F.col(col).cast(typ))
    return df

def write_silver(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.silver.{name}"))
    print(f"silver.{name}: {df.count()} rows, {len(df.columns)} cols")

In [7]:
enc = read_bronze("encounters")

enc = (enc
    .withColumnRenamed("id", "encounter_id")
    .withColumnRenamed("patient", "patient_id")          # <- foreign key to patients
    .withColumnRenamed("organization", "organization_id")
    .withColumnRenamed("provider", "provider_id")
    .withColumnRenamed("payer", "payer_id")
    .withColumnRenamed("encounterclass", "encounter_class")
    .withColumnRenamed("reasoncode", "reason_code")
    .withColumnRenamed("reasondescription", "reason_description")
    # parse timestamps ONLY to derive year + duration; raw values won't survive
    .withColumn("start_ts", F.to_timestamp("start"))
    .withColumn("stop_ts",  F.to_timestamp("stop"))
    .withColumn("start_year", F.year("start_ts"))
    .withColumn("length_of_stay_hours",
                F.round((F.col("stop_ts").cast("long") - F.col("start_ts").cast("long")) / 3600.0, 2))
)

enc = apply_casts(enc, {
    "base_encounter_cost": "decimal(12,2)",
    "total_claim_cost":    "decimal(12,2)",
    "payer_coverage":      "decimal(12,2)",
})

silver_enc = enc.select(
    "encounter_id", "patient_id",
    "organization_id", "provider_id", "payer_id",
    "encounter_class",
    "code", "description",                 # SNOMED codes stay STRING
    "reason_code", "reason_description",
    "start_year", "length_of_stay_hours",  # Safe Harbor: year + interval, no raw dates
    "base_encounter_cost", "total_claim_cost", "payer_coverage",
    "_ingested_at",
)

write_silver(silver_enc, "encounters")

silver.encounters: 5906 rows, 16 cols


In [8]:
orphans = (spark.table(f"{CATALOG}.silver.encounters")
    .join(spark.table(f"{CATALOG}.silver.patients"), "patient_id", "left_anti")
    .count())
print(f"orphan encounters with no matching patient: {orphans}")   # expect 0

orphan encounters with no matching patient: 0


In [9]:
%sql
SELECT p.race, p.gender,
       COUNT(*)                        AS encounters,
       ROUND(AVG(e.total_claim_cost),2) AS avg_claim_cost
FROM workspace.silver.encounters e
JOIN workspace.silver.patients p USING (patient_id)
GROUP BY p.race, p.gender
ORDER BY encounters DESC;

race,gender,encounters,avg_claim_cost
white,F,2255,3914.84
white,M,1642,2576.00
black,F,883,1956.83
asian,F,702,2004.81
asian,M,160,1705.71
black,M,104,1689.59
hawaiian,F,73,2473.80
native,F,69,1738.36
hawaiian,M,18,1277.07


In [10]:
# ---------- conditions ----------
cond = read_bronze("conditions")
cond = (cond
    .withColumnRenamed("patient", "patient_id")
    .withColumnRenamed("encounter", "encounter_id")
    .withColumn("start_year", F.year(F.to_date("start")))
    .withColumn("stop_year",  F.year(F.to_date("stop"))))
silver_cond = cond.select(
    "patient_id", "encounter_id",
    "code", "description",          # SNOMED, stays string
    "start_year", "stop_year",
    "_ingested_at")
write_silver(silver_cond, "conditions")

# ---------- medications ----------
med = read_bronze("medications")
med = (med
    .withColumnRenamed("patient", "patient_id")
    .withColumnRenamed("encounter", "encounter_id")
    .withColumnRenamed("payer", "payer_id")
    .withColumnRenamed("reasoncode", "reason_code")
    .withColumnRenamed("reasondescription", "reason_description")
    .withColumn("start_year", F.year(F.to_date("start")))
    .withColumn("stop_year",  F.year(F.to_date("stop"))))
med = apply_casts(med, {
    "base_cost": "decimal(12,2)",
    "payer_coverage": "decimal(12,2)",
    "totalcost": "decimal(12,2)",
    "dispenses": "int"})
silver_med = med.select(
    "patient_id", "encounter_id", "payer_id",
    "code", "description",          # RxNorm, stays string
    "reason_code", "reason_description",
    "start_year", "stop_year",
    "base_cost", "payer_coverage",
    F.col("totalcost").alias("total_cost"),
    "dispenses",
    "_ingested_at")
write_silver(silver_med, "medications")

# ---------- procedures ----------
proc = read_bronze("procedures")
proc = (proc
    .withColumnRenamed("patient", "patient_id")
    .withColumnRenamed("encounter", "encounter_id")
    .withColumnRenamed("reasoncode", "reason_code")
    .withColumnRenamed("reasondescription", "reason_description")
    .withColumn("start_year", F.year(F.to_date("start"))))
proc = apply_casts(proc, {"base_cost": "decimal(12,2)"})
silver_proc = proc.select(
    "patient_id", "encounter_id",
    "code", "description",          # SNOMED/CPT, stays string
    "reason_code", "reason_description",
    "start_year", "base_cost",
    "_ingested_at")
write_silver(silver_proc, "procedures")

# ---------- observations (the tricky one) ----------
obs = read_bronze("observations")
obs = (obs
    .withColumnRenamed("patient", "patient_id")
    .withColumnRenamed("encounter", "encounter_id")
    .withColumn("obs_year", F.year(F.to_date("date")))
    # VALUE mixes numeric labs and text results; try_cast keeps numbers,
    # returns null for text instead of erroring (ANSI mode is on)
    .withColumn("value_numeric", F.expr("try_cast(value AS double)")))

silver_obs = obs.select(
    "patient_id", "encounter_id",
    "category", "code", "description",
    F.col("value").alias("value_raw"),
    "value_numeric",
    "units", "type",
    "obs_year",
    "_ingested_at")
write_silver(silver_obs, "observations")

silver.conditions: 3975 rows, 7 cols
silver.medications: 4096 rows, 14 cols
silver.procedures: 17002 rows, 9 cols
silver.observations: 73883 rows, 11 cols


In [11]:
%sql
SELECT description, value_raw, value_numeric, units
FROM workspace.silver.observations
WHERE value_numeric IS NOT NULL
LIMIT 5;

SELECT description, value_raw, value_numeric
FROM workspace.silver.observations
WHERE value_numeric IS NULL
LIMIT 5;

description,value_raw,value_numeric
Tobacco smoking status,Never smoked tobacco (finding),null
Within the last year have you been afraid of your partner or ex-partner,No,null
Do you feel physically and emotionally safe where you currently live [PRAPARE],Yes,null
Are you a refugee,No,null
Have you spent more than 2 nights in a row in a jail prison detention center or juvenile correctional facility in past 1 year [PRAPARE],No,null


## Gold layer — patient feature & cohort tables

Collapse the six Silver tables into one row per patient via a shared aggregation base,
then branch into two consumer-shaped tables: `gold.patient_features` (wide, numeric,
model-ready — protected attributes deliberately excluded) and `gold.patient_cohort`
(human-readable dimensions + measures for dashboards).


In [12]:
from pyspark.sql import functions as F

CATALOG = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

p   = spark.table(f"{CATALOG}.silver.patients")
enc = spark.table(f"{CATALOG}.silver.encounters")
con = spark.table(f"{CATALOG}.silver.conditions")
med = spark.table(f"{CATALOG}.silver.medications")
pro = spark.table(f"{CATALOG}.silver.procedures")
obs = spark.table(f"{CATALOG}.silver.observations")

# ---- per-domain rollups: one row per patient_id ----
enc_agg = enc.groupBy("patient_id").agg(
    F.count("*").alias("encounter_count"),
    F.countDistinct("encounter_class").alias("distinct_encounter_classes"),
    F.round(F.sum("total_claim_cost"), 2).alias("total_encounter_cost"),
    F.round(F.avg("length_of_stay_hours"), 2).alias("avg_los_hours"),
    F.min("start_year").alias("first_encounter_year"),
    F.max("start_year").alias("last_encounter_year"),
)

con_agg = con.groupBy("patient_id").agg(
    F.count("*").alias("condition_count"),
    F.countDistinct("code").alias("distinct_condition_count"),
    # disease flags by description match -> int 0/1 (max = "any row matched")
    F.max(F.lower(F.col("description")).contains("diabetes").cast("int")).alias("has_diabetes"),
    F.max(F.lower(F.col("description")).contains("hypertension").cast("int")).alias("has_hypertension"),
    F.max(F.lower(F.col("description")).contains("asthma").cast("int")).alias("has_asthma"),
)

med_agg = med.groupBy("patient_id").agg(
    F.count("*").alias("medication_count"),
    F.countDistinct("code").alias("distinct_medication_count"),
    F.round(F.sum("total_cost"), 2).alias("total_medication_cost"),
)

pro_agg = pro.groupBy("patient_id").agg(
    F.count("*").alias("procedure_count"),
    F.countDistinct("code").alias("distinct_procedure_count"),
    F.round(F.sum("base_cost"), 2).alias("total_procedure_cost"),
)

obs_agg = obs.groupBy("patient_id").agg(
    F.count("*").alias("observation_count"),
)

# ---- shared base: patients + all rollups, one row per patient ----
base = (p
    .join(enc_agg, "patient_id", "left")
    .join(con_agg, "patient_id", "left")
    .join(med_agg, "patient_id", "left")
    .join(pro_agg, "patient_id", "left")
    .join(obs_agg, "patient_id", "left"))

# patients with no rows in a domain -> 0, not null (counts/costs/flags only;
# NOT birth_year, which is legitimately null for the 90+ cohort)
fill_zero = [
    "encounter_count","distinct_encounter_classes","total_encounter_cost",
    "condition_count","distinct_condition_count",
    "has_diabetes","has_hypertension","has_asthma",
    "medication_count","distinct_medication_count","total_medication_cost",
    "procedure_count","distinct_procedure_count","total_procedure_cost",
    "observation_count",
]
base = base.fillna(0, subset=fill_zero)

# ======== GOLD TABLE 1: ML feature table (wide, numeric, model-ready) ========
ml = base.select(
    "patient_id",
    "age",
    F.col("is_deceased").cast("int").alias("is_deceased"),
    (F.col("gender") == "M").cast("int").alias("is_male"),
    "income",
    F.col("healthcare_expenses").cast("double").alias("healthcare_expenses"),
    F.col("healthcare_coverage").cast("double").alias("healthcare_coverage"),
    "encounter_count", "distinct_encounter_classes", "total_encounter_cost", "avg_los_hours",
    "condition_count", "distinct_condition_count",
    "has_diabetes", "has_hypertension", "has_asthma",
    "medication_count", "distinct_medication_count", "total_medication_cost",
    "procedure_count", "distinct_procedure_count", "total_procedure_cost",
    "observation_count",
)
(ml.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.gold.patient_features"))
print(f"gold.patient_features: {ml.count()} rows, {len(ml.columns)} cols")

# ======== GOLD TABLE 2: cohort analytics (dimensions + measures, human-readable) ========
age_band = (F.when(F.col("age") < 18, "0-17")
             .when(F.col("age") < 30, "18-29")
             .when(F.col("age") < 45, "30-44")
             .when(F.col("age") < 65, "45-64")
             .when(F.col("age") < 80, "65-79")
             .otherwise("80+"))

cohort = base.select(
    "patient_id",
    age_band.alias("age_band"),
    "gender", "race", "ethnicity", "marital",
    "state", "zip3",
    "is_deceased",
    F.col("has_diabetes").cast("boolean").alias("has_diabetes"),
    F.col("has_hypertension").cast("boolean").alias("has_hypertension"),
    F.col("has_asthma").cast("boolean").alias("has_asthma"),
    "encounter_count", "total_encounter_cost",
    "condition_count", "distinct_condition_count",
    "medication_count", "total_medication_cost",
    "procedure_count", "total_procedure_cost",
    F.round(F.col("total_encounter_cost") + F.col("total_medication_cost")
            + F.col("total_procedure_cost"), 2).alias("total_healthcare_cost"),
)
(cohort.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOG}.gold.patient_cohort"))
print(f"gold.patient_cohort: {cohort.count()} rows, {len(cohort.columns)} cols")

gold.patient_features: 112 rows, 23 cols
gold.patient_cohort: 112 rows, 21 cols


In [13]:
%sql
SELECT patient_id, age, is_male, encounter_count, distinct_condition_count,
       has_diabetes, has_hypertension, total_encounter_cost, total_medication_cost
FROM workspace.gold.patient_features
ORDER BY total_encounter_cost DESC
LIMIT 10;

patient_id,age,is_male,encounter_count,distinct_condition_count,has_diabetes,has_hypertension,total_encounter_cost,total_medication_cost
b2f88ecf-0353-0ddb-37da-4ac9e4434af2,23,0,132,22,0,0,788449.48,82407.29
91592d7b-43f9-b288-0255-e44789759a0a,37,0,70,18,0,0,717279.52,24990.42
37c4b8d4-e6af-ab4d-9924-9ff159a9dc6e,23,0,111,23,0,0,667173.09,200333.81
95b90826-c423-8044-d172-514a67037f8e,22,1,88,16,0,0,597117.17,121231.94
edcaee04-2b9d-47a5-5fb5-b68ca4bafbd5,62,0,580,33,1,1,579465.00,51067.39
7a58a941-bd27-086e-d622-38e5917d0df5,28,0,63,22,0,0,531232.24,813.54
0c1e1add-0fb2-9ad8-d9fb-22dd45ab41e3,62,0,518,39,1,1,524886.37,6118.19
f1f05592-2511-92df-6f3e-c64ac3152b26,43,0,66,23,1,0,454701.72,53213.22
e9de4431-5532-6bbd-73fe-c50596f7a834,55,0,74,28,0,1,452780.53,121939.66
8b39d546-3e1e-ef33-faa9-061dd7d8dc62,43,0,57,21,1,0,446439.41,62012.66


In [14]:
%sql
SELECT age_band,
       has_diabetes,
       COUNT(*)                              AS patients,
       ROUND(AVG(total_healthcare_cost), 0)  AS avg_total_cost
FROM workspace.gold.patient_cohort
GROUP BY age_band, has_diabetes
ORDER BY age_band, has_diabetes;

age_band,has_diabetes,patients,avg_total_cost
0-17,false,15,54187
18-29,false,14,649992
18-29,true,3,93013
30-44,false,10,290313
30-44,true,8,450722
45-64,false,12,342424
45-64,true,32,317369
65-79,false,2,235510
65-79,true,13,454962
80+,false,1,167701


## SDTM mapping — real-world data → CDISC SDTMIG v3.4

Map the Silver tables into SDTM-conformant domains following the observational-study
pattern (CDISC "Considerations for SDTM Implementation in Observational Studies"). One
domain per general observation class: **DM** (Special Purpose), **VS** + **LB**
(Findings), **CM** + **PR** (Interventions), **MH** (Events). Controlled Terminology is
applied for SEX/RACE/ETHNIC and vitals test codes; the LOINC lab long-tail is left in a
documented fallback (see `VALIDATION.md`). Year-only `--DTC` dates are valid partial
ISO 8601 — the Safe Harbor output composes cleanly with SDTM date rules.


In [15]:
from pyspark.sql import functions as F
from itertools import chain

CATALOG = "workspace"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.sdtm")

STUDYID = "RWD2026"   # notional study frame for the RWD-to-SDTM mapping

def ct_map(d):                                   # build a CDISC CT lookup
    return F.create_map([F.lit(x) for x in chain(*d.items())])

# CDISC Controlled Terminology: Synthea values -> CDISC CT submission values
RACE_CT = ct_map({
    "white":    "WHITE",
    "black":    "BLACK OR AFRICAN AMERICAN",
    "asian":    "ASIAN",
    "native":   "AMERICAN INDIAN OR ALASKA NATIVE",
    "hawaiian": "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER",
    "other":    "OTHER",
})
ETHNIC_CT = ct_map({
    "hispanic":    "HISPANIC OR LATINO",
    "nonhispanic": "NOT HISPANIC OR LATINO",
})

p   = spark.table(f"{CATALOG}.silver.patients")
enc = spark.table(f"{CATALOG}.silver.encounters")

# reference start = first encounter year (year-only IS valid ISO 8601 for --DTC)
rfst = enc.groupBy("patient_id").agg(F.min("start_year").alias("rfst_year"))

dm = (p.join(rfst, "patient_id", "left")
    .withColumn("STUDYID", F.lit(STUDYID))
    .withColumn("DOMAIN",  F.lit("DM"))
    .withColumn("USUBJID", F.concat_ws("-", F.lit(STUDYID), F.col("patient_id")))
    .withColumn("SUBJID",  F.col("patient_id"))
    .withColumn("SITEID",  F.lit("01"))                      # single notional site
    .withColumn("RFSTDTC", F.col("rfst_year").cast("string"))  # ISO 8601 year
    .withColumn("DTHFL",   F.when(F.col("is_deceased"), F.lit("Y")))
    .withColumn("DTHDTC",  F.when(F.col("is_deceased"), F.col("death_year").cast("string")))
    .withColumn("AGE",     F.col("age"))
    .withColumn("AGEU",    F.lit("YEARS"))
    .withColumn("SEX",     F.coalesce(F.col("gender"), F.lit("U")))          # CT: M/F/U
    .withColumn("RACE",    F.coalesce(RACE_CT[F.col("race")], F.lit("UNKNOWN")))
    .withColumn("ETHNIC",  F.coalesce(ETHNIC_CT[F.col("ethnicity")], F.lit("NOT REPORTED")))
    .withColumn("ARMCD",   F.lit("OBS"))                     # observational cohort
    .withColumn("ARM",     F.lit("Observational Cohort"))
    .withColumn("COUNTRY", F.lit("USA"))                     # ISO 3166-1 alpha-3
    .select("STUDYID","DOMAIN","USUBJID","SUBJID","SITEID","RFSTDTC",
            "DTHFL","DTHDTC","AGE","AGEU","SEX","RACE","ETHNIC",
            "ARMCD","ARM","COUNTRY"))

(dm.write.format("delta").mode("overwrite").option("overwriteSchema","true")
   .saveAsTable(f"{CATALOG}.sdtm.dm"))
print(f"sdtm.dm: {dm.count()} subjects, {len(dm.columns)} variables")
dm.show(5, truncate=False)

sdtm.dm: 112 subjects, 16 variables
+-------+------+--------------------------------------------+------------------------------------+------+-------+-----+------+---+-----+---+-------------------------+----------------------+-----+--------------------+-------+
|STUDYID|DOMAIN|USUBJID                                     |SUBJID                              |SITEID|RFSTDTC|DTHFL|DTHDTC|AGE|AGEU |SEX|RACE                     |ETHNIC                |ARMCD|ARM                 |COUNTRY|
+-------+------+--------------------------------------------+------------------------------------+------+-------+-----+------+---+-----+---+-------------------------+----------------------+-----+--------------------+-------+
|RWD2026|DM    |RWD2026-a35dfdc9-1d97-63e0-a0b6-3e97b6fd5fe1|a35dfdc9-1d97-63e0-a0b6-3e97b6fd5fe1|01    |2010   |NULL |NULL  |30 |YEARS|M  |WHITE                    |NOT HISPANIC OR LATINO|OBS  |Observational Cohort|USA    |
|RWD2026|DM    |RWD2026-5393ea7f-1fea-4064-9df4-460a2f662d07|539

In [16]:
%sql
SELECT USUBJID, AGE, SEX, RACE, ETHNIC, RFSTDTC, DTHFL, ARM
FROM workspace.sdtm.dm
ORDER BY DTHFL DESC NULLS LAST
LIMIT 10;

USUBJID,AGE,SEX,RACE,ETHNIC,RFSTDTC,DTHFL,ARM
RWD2026-cf2beebb-f66f-7684-6099-312cbb96dbc6,48,M,WHITE,NOT HISPANIC OR LATINO,1994,Y,Observational Cohort
RWD2026-0c1e1add-0fb2-9ad8-d9fb-22dd45ab41e3,62,F,ASIAN,NOT HISPANIC OR LATINO,1968,Y,Observational Cohort
RWD2026-59d21b6d-ef06-1497-4b9c-19918733d3f6,35,F,ASIAN,NOT HISPANIC OR LATINO,1987,Y,Observational Cohort
RWD2026-078df27f-f693-a4cf-9f23-872b9e3ab3d2,79,F,WHITE,HISPANIC OR LATINO,1959,Y,Observational Cohort
RWD2026-6c8f0d25-5e38-ce5c-2add-8cb09ec415de,5,F,WHITE,HISPANIC OR LATINO,1971,Y,Observational Cohort
RWD2026-288427c5-ee03-f47e-6872-def5772f427c,69,M,WHITE,HISPANIC OR LATINO,1960,Y,Observational Cohort
RWD2026-5393ea7f-1fea-4064-9df4-460a2f662d07,12,M,WHITE,NOT HISPANIC OR LATINO,1976,Y,Observational Cohort
RWD2026-7a58a941-bd27-086e-d622-38e5917d0df5,28,F,WHITE,NOT HISPANIC OR LATINO,1972,Y,Observational Cohort
RWD2026-86919c2e-6fcc-4756-2a76-c0e31e732109,58,M,WHITE,NOT HISPANIC OR LATINO,1976,Y,Observational Cohort
RWD2026-e9de4431-5532-6bbd-73fe-c50596f7a834,55,F,BLACK OR AFRICAN AMERICAN,NOT HISPANIC OR LATINO,1973,Y,Observational Cohort


In [17]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
STUDYID = "RWD2026"

# real CDISC VS controlled-terminology codes for the common vitals (keyed on LOINC)
VS_TESTCD = ct_map({
    "8302-2":  "HEIGHT", "29463-7": "WEIGHT", "39156-5": "BMI",
    "8480-6":  "SYSBP",  "8462-4":  "DIABP",  "8867-4":  "HR",
    "9279-1":  "RESP",   "8310-5":  "TEMP",   "2708-6":  "SPO2", "59408-5": "SPO2",
    "72514-3": "PAIN",   "725143":  "PAIN",   # pain rating -> CT code, both code forms
})
# illustrative LB crosswalk; the long tail falls back (see SDRG note)
LB_TESTCD = ct_map({
    "2339-0": "GLUC", "718-7": "HGB", "4544-3": "HCT",
    "2160-0": "CREAT", "3094-0": "BUN", "2951-2": "SODIUM",
})

def build_findings(category_value, domain, testcd_map):
    o = (spark.table(f"{CATALOG}.silver.observations")
         .filter(F.col("category") == category_value)
         .withColumn("USUBJID", F.concat_ws("-", F.lit(STUDYID), F.col("patient_id"))))

    # TESTCD from crosswalk; fallback = first 8 alphanumerics of the code (placeholder)
    testcd = F.coalesce(
        testcd_map[F.col("code")],
        F.upper(F.substring(F.regexp_replace(F.col("code"), "[^A-Za-z0-9]", ""), 1, 8)))

    o = (o
        .withColumn("STUDYID", F.lit(STUDYID))
        .withColumn("DOMAIN",  F.lit(domain))
        .withColumn(f"{domain}TESTCD", testcd)
        .withColumn(f"{domain}TEST",   F.col("description"))
        .withColumn(f"{domain}ORRES",  F.col("value_raw"))
        .withColumn(f"{domain}ORRESU", F.col("units"))
        .withColumn(f"{domain}STRESC", F.col("value_raw"))
        .withColumn(f"{domain}STRESN", F.col("value_numeric"))   # numeric only, null for text
        .withColumn(f"{domain}STRESU", F.col("units"))
        .withColumn(f"{domain}DTC",    F.col("obs_year").cast("string")))   # ISO 8601 year

    # --SEQ: unique sequence within each subject
    w = Window.partitionBy("USUBJID").orderBy(f"{domain}DTC", f"{domain}TESTCD")
    o = o.withColumn(f"{domain}SEQ", F.row_number().over(w))

    return o.select(
        "STUDYID", "DOMAIN", "USUBJID", f"{domain}SEQ",
        f"{domain}TESTCD", f"{domain}TEST",
        f"{domain}ORRES", f"{domain}ORRESU",
        f"{domain}STRESC", f"{domain}STRESN", f"{domain}STRESU",
        f"{domain}DTC")

vs = build_findings("vital-signs", "VS", VS_TESTCD)
lb = build_findings("laboratory",  "LB", LB_TESTCD)

(vs.write.format("delta").mode("overwrite").option("overwriteSchema","true")
   .saveAsTable(f"{CATALOG}.sdtm.vs"))
print(f"sdtm.vs: {vs.count()} records, {len(vs.columns)} variables")

(lb.write.format("delta").mode("overwrite").option("overwriteSchema","true")
   .saveAsTable(f"{CATALOG}.sdtm.lb"))
print(f"sdtm.lb: {lb.count()} records, {len(lb.columns)} variables")

sdtm.vs: 13875 records, 12 variables
sdtm.lb: 30470 records, 12 variables


In [18]:
%sql
SELECT USUBJID, VSSEQ, VSTESTCD, VSTEST, VSORRES, VSORRESU, VSSTRESN
FROM workspace.sdtm.vs
WHERE VSSTRESN IS NOT NULL
ORDER BY USUBJID, VSSEQ
LIMIT 15;

USUBJID,VSSEQ,VSTESTCD,VSTEST,VSORRES,VSORRESU,VSSTRESN
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,1,BMI,Body mass index (BMI) [Ratio],29.4,kg/m2,29.4
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,2,DIABP,Diastolic Blood Pressure,72.0,mm[Hg],72.0
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,3,HEIGHT,Body Height,177.2,cm,177.2
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,4,HR,Heart rate,96.0,/min,96.0
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,5,PAIN,Pain severity - 0-10 verbal numeric rating [Score] - Reported,1.0,{score},1.0
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,6,RESP,Respiratory rate,14.0,/min,14.0
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,7,SYSBP,Systolic Blood Pressure,132.0,mm[Hg],132.0
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,8,WEIGHT,Body Weight,92.3,kg,92.3
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,9,BMI,Body mass index (BMI) [Ratio],29.4,kg/m2,29.4
RWD2026-006232dd-9560-5c9a-1430-c5ceaacb45c3,10,DIABP,Diastolic Blood Pressure,72.0,mm[Hg],72.0


In [19]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
STUDYID = "RWD2026"

def usubjid():
    return F.concat_ws("-", F.lit(STUDYID), F.col("patient_id"))

def add_seq(df, domain):
    w = Window.partitionBy("USUBJID").orderBy(f"{domain}STDTC", f"{domain}DECOD")
    return df.withColumn(f"{domain}SEQ", F.row_number().over(w))

# ---------- CM: Concomitant Medications (Interventions) ----------
cm = (spark.table(f"{CATALOG}.silver.medications")
    .withColumn("USUBJID", usubjid())
    .withColumn("STUDYID", F.lit(STUDYID))
    .withColumn("DOMAIN",  F.lit("CM"))
    .withColumn("CMTRT",   F.col("description"))          # topic: the treatment
    .withColumn("CMDECOD", F.col("code"))                 # RxNorm code
    .withColumn("CMINDC",  F.col("reason_description"))   # indication
    .withColumn("CMSTDTC", F.col("start_year").cast("string"))
    .withColumn("CMENDTC", F.col("stop_year").cast("string")))
cm = add_seq(cm, "CM").select(
    "STUDYID","DOMAIN","USUBJID","CMSEQ",
    "CMTRT","CMDECOD","CMINDC","CMSTDTC","CMENDTC")

# ---------- PR: Procedures (Interventions) ----------
pr = (spark.table(f"{CATALOG}.silver.procedures")
    .withColumn("USUBJID", usubjid())
    .withColumn("STUDYID", F.lit(STUDYID))
    .withColumn("DOMAIN",  F.lit("PR"))
    .withColumn("PRTRT",   F.col("description"))
    .withColumn("PRDECOD", F.col("code"))                 # SNOMED/CPT
    .withColumn("PRINDC",  F.col("reason_description"))
    .withColumn("PRSTDTC", F.col("start_year").cast("string")))
pr = add_seq(pr, "PR").select(
    "STUDYID","DOMAIN","USUBJID","PRSEQ",
    "PRTRT","PRDECOD","PRINDC","PRSTDTC")

# ---------- MH: Medical History (Events) ----------
mh = (spark.table(f"{CATALOG}.silver.conditions")
    .withColumn("USUBJID", usubjid())
    .withColumn("STUDYID", F.lit(STUDYID))
    .withColumn("DOMAIN",  F.lit("MH"))
    .withColumn("MHTERM",  F.col("description"))          # topic: the term
    .withColumn("MHDECOD", F.col("code"))                 # SNOMED
    .withColumn("MHSTDTC", F.col("start_year").cast("string"))
    .withColumn("MHENDTC", F.col("stop_year").cast("string")))
mh = add_seq(mh, "MH").select(
    "STUDYID","DOMAIN","USUBJID","MHSEQ",
    "MHTERM","MHDECOD","MHSTDTC","MHENDTC")

for df, name in [(cm,"cm"), (pr,"pr"), (mh,"mh")]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema","true")
       .saveAsTable(f"{CATALOG}.sdtm.{name}"))
    print(f"sdtm.{name}: {df.count()} records, {len(df.columns)} variables")

sdtm.cm: 4096 records, 9 variables
sdtm.pr: 17002 records, 8 variables
sdtm.mh: 3975 records, 8 variables


In [20]:
%sql
SELECT 'CM' d, COUNT(*) n FROM workspace.sdtm.cm
UNION ALL SELECT 'PR', COUNT(*) FROM workspace.sdtm.pr
UNION ALL SELECT 'MH', COUNT(*) FROM workspace.sdtm.mh;

d,n
CM,4096
PR,17002
MH,3975


In [21]:
CATALOG = "workspace"
domains = ["dm", "vs", "lb", "cm", "pr", "mh"]

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.sdtm.export")

for d in domains:
    (spark.table(f"{CATALOG}.sdtm.{d}")
        .coalesce(1)                      # one file per domain, not many shards
        .write.format("csv").mode("overwrite").option("header", "true")
        .save(f"/Volumes/{CATALOG}/sdtm/export/{d}"))
print("exported:", domains)

exported: ['dm', 'vs', 'lb', 'cm', 'pr', 'mh']
